# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jait-mukkamalla/flyrank-internship-ml/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

***Lane 2: Refresh / Content Opportunity Scoring***

**Question**:
Which pages should be reviewed first for refresh, expansion, protection, pruning, or monitoring?

**Why**:
This lane of work was the most interesting for me due to the added complexities and intricacies of the task(s) at hand. Creating a model that is capable of determining opportunites for improvement and the urgency at which changes should be made is a very important aspect of the SEO optimization work done at FlyRank AI (in my opinion). I was enticed by the opportunity to work with and learn about more complex AI model methods, such as gradient boosting and random forest, and the various ways adding complexity can help improve the performance of ML models.

In [39]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The question: decision, action, cost of a wrong call

**Improved decision**:
A ML model with better refresh & content opportunity scoring identifies which individual pages should be prioritized — and what action to take (refresh, expand, protect, prune, or monitor) — rather than treating a whole site as uniformly needing attention.

**Who uses it**:
Content strategists / SEO analysts who build the prioritized action queue each cycle, and use the model's output to decide where to spend limited writing, editing, and dev hours across a client's page inventory. Client-facing account managers may also use the scores to explain and defend recommendations to clients.

**Cost of being wrong**:
The cost differs by action type, since each error type wastes a different resource:
- **Refresh recommended on a page that didn't need it**: wasted content-team hours on low-impact edits, while genuinely decaying pages keep losing rankings/traffic unattended.
- **Expand recommended incorrectly**: budget spent building out a page that won't convert the added investment into traffic or rankings gains.
- **Protect missed** (a strong page is left unmonitored and starts declining): silent traffic/revenue loss that isn't caught until it's already significant, since "protect" pages are usually the ones with the most to lose.
- **Prune recommended on a page that still has value**: lost traffic/backlinks/rankings equity that's hard to recover once a page is removed or de-indexed.
- **Monitor mislabeled** (a page needing urgent action is bucketed as "just watch it"): the highest-cost failure mode — real opportunity or risk goes unaddressed for a full cycle before anyone re-reviews it.

Across all five actions, the common thread is ***misallocated attention***: every hour spent on a wrongly-flagged page is an hour not spent on a page that actually needed it, and the compounding cost is a client's overall content ROI trending down without a clear signal pointing to why.

In [40]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

**Computed numbers**:
- 54.2% of pages are declining (that is 16,262 declining out of 30,000 eligible pages)
- The worst performing freshness tier declines at 1.3x the rate of the best performing tier
- Declining pages average 12.5% fewer sessions than stable/growing pages

**Key takeaways**:
A large share of content is declining, and the decline isn't random — it's concentrated and comes with a measurable engagement gap, which means a scoring model has something real to prioritize and learn from. Multiple features across the data signify the decline of pages, so a multi-feature ML model will be needed here, hence the aforementioned complexity of using model methods like random forest and gradient boosting.

In [41]:
import pandas as pd

url = "https://raw.githubusercontent.com/jait-mukkamalla/flyrank-internship-ml/refs/heads/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)
print("Rows:", len(df))

# LABEL: convert trend direction to a numeric value that allows us to perform calculations and obtain metrics like mean
df["is_down"] = (df["trend_direction"] == "down").astype(int)


# NUMBER 1: Overall decline rate - "What % of eligible pages are trending down?"
overall_decline_rate = df["is_down"].mean()  # mean of 0/1 column = % that are 1
print(f"\n[1] Overall decline rate: {overall_decline_rate:.1%}")
print(f"    ({df['is_down'].sum():,} declining out of {len(df):,} eligible pages)")


# NUMBER 2: Decline rate by freshness_tier - "Is the problem spread evenly, or concentrated?"
decline_by_tier = (
    df.groupby("freshness_tier")["is_down"]
    .agg(decline_rate="mean", n="count")
    .sort_values("decline_rate", ascending=False)
    .reset_index()
)

# --- do the numeric math FIRST, while decline_rate is still a float ---
worst = decline_by_tier["decline_rate"].max()
best = decline_by_tier["decline_rate"].min()
ratio_text = f"Worst tier declines at {worst / best:.1f}x the rate of the best tier"

# --- THEN convert to a display string for printing ---
decline_by_tier["decline_rate"] = (decline_by_tier["decline_rate"] * 100).round(1).astype(str) + "%"

print("\n[2] Decline rate by freshness_tier:")
print(decline_by_tier.to_string(index=False))
print(f"\n    {ratio_text}")


# NUMBER 3: Engagement gap between declining and non-declining pages - "Is there signal the model could actually learn from?"
engagement_by_label = (
    df.groupby("is_down")["sessions_90d"]
    .mean().round(3)
    .rename(index={0: "not_down", 1: "down"})
    .rename_axis(None)
)

print("\n[3] Average sessions_90d by trend label:")
print(engagement_by_label.to_string())

pct_gap = (
    (engagement_by_label["not_down"] - engagement_by_label["down"])
    / engagement_by_label["not_down"]
)
print(f"\n    Declining pages average {pct_gap:.1%} fewer sessions than stable/growing pages")

Rows: 30000

[1] Overall decline rate: 54.2%
    (16,262 declining out of 30,000 eligible pages)

[2] Decline rate by freshness_tier:
freshness_tier decline_rate     n
        91-180        61.1%  9171
         31-90        58.9%   175
          0-30        51.1% 20480
          181+        47.1%   174

    Worst tier declines at 1.3x the rate of the best tier

[3] Average sessions_90d by trend label:
not_down    39.762
down        34.789

    Declining pages average 12.5% fewer sessions than stable/growing pages


## 4. Careful words: what I can and can't claim

**What this work will be able to say:**
- *Observed*: Using FlyRank's warehouse dataset of historical page-level content and performance signals, which pages exhibited patterns of decline that preceded refresh, expansion, prune, protect, or monitor actions, and what happened to key metrics afterward.
- *Directional*: Given a page's current signals, how closely it resembles historical pages associated with a given action, and a relative ranking of urgency/opportunity across a client's page inventory — a prioritized shortlist, not a certainty.
- *Decision-support*: A model-generated queue of "review this first" pages with a suggested action, intended to focus strategist attention — the model surfaces and ranks candidates; a human still applies judgment about client context before acting.

**What this work will never say:**
- *Causal proof*: Even at 79M rows, this is observational, not experimental, data — no randomized assignment of "refresh vs. don't refresh" exists. The model can show correlation between signals and historical outcomes, not that taking an action *caused* the outcome.
- *"Predicting Google"*: The model pattern-matches against historical content/performance signals — it does not model or forecast Google's ranking systems directly.
- *Guaranteed outcomes*: A high opportunity score reflects resemblance to pages that historically benefited from an action, not a guarantee that this specific page will respond the same way.

In [42]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.